# ViSceT5 — Finetune từ checkpoint PRETRAIN
Chạy tuần tự. Chỉ cần điền **HF token** ở cell cấu hình.

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
!git pull

In [ ]:
!bash setup.sh
# Nếu Colab báo cần restart: Runtime > Restart, rồi chạy tiếp TỪ cell cấu hình
# (KHÔNG cần chạy lại 2 cell clone/setup).

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_xxx'          # <== ĐIỀN token HF của bạn
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-1epoch'   # repo chứa MODEL ĐÃ PRETRAIN
HF_FINETUNE_REPO = 'Kus669/ViSceT5-finetune'          # repo sẽ lưu model finetune

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### Chọn + tải checkpoint PRETRAIN CUỐI CÙNG
Ưu tiên model ở **gốc repo** (bản `save_model` cuối = tốt nhất do load_best_model_at_end). Chỉ tải root, KHÔNG kéo checkpoint trung gian (đỡ ~GB).

In [ ]:
import os, re
from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=os.environ['HF_TOKEN'])
files = api.list_repo_files(HF_PRETRAIN_REPO, repo_type='model')

if 'model.safetensors' in files:
    subdir = ''
    print('=> Dùng model ở gốc repo (final save_model / best).')
else:
    steps = [int(m.group(1)) for f in files
             for m in [re.match(r'checkpoint-(\d+)/model\.safetensors$', f)] if m]
    assert steps, 'Khong tim thay model.safetensors nao trong repo!'
    subdir = f'checkpoint-{max(steps)}'
    print('=> Dùng checkpoint moi nhat:', subdir)

pat = (subdir + '/') if subdir else ''
snapshot_download(
    repo_id=HF_PRETRAIN_REPO, repo_type='model', local_dir='/content/pretrain_dl',
    allow_patterns=[pat+'config.json', pat+'generation_config.json', pat+'model.safetensors',
                    pat+'spiece.model', pat+'tokenizer*.json', pat+'special_tokens_map.json',
                    pat+'tokenizer_config.json'],
    ignore_patterns=(['checkpoint-*/**'] if subdir == '' else None),
    token=os.environ['HF_TOKEN'],
)
PRETRAIN_DIR = os.path.join('/content/pretrain_dl', subdir)
print('PRETRAIN_DIR =', PRETRAIN_DIR)
assert os.path.exists(os.path.join(PRETRAIN_DIR, 'model.safetensors')), 'Thieu model.safetensors!'

### (Xác minh) Model lấy về ứng với epoch/step nào
Đọc trainer_state của checkpoint mới nhất để chắc chắn đã train đủ epoch và biết best checkpoint.

In [ ]:
import json
from huggingface_hub import hf_hub_download
_steps = sorted({int(m.group(1)) for f in files for m in [re.match(r'checkpoint-(\d+)/', f)] if m})
print('Checkpoints tren repo:', _steps)
if _steps:
    _ts = hf_hub_download(HF_PRETRAIN_REPO, f'checkpoint-{_steps[-1]}/trainer_state.json',
                          repo_type='model', token=os.environ['HF_TOKEN'])
    _st = json.load(open(_ts))
    print('Moi nhat:', _steps[-1], '| epoch =', round(_st.get('epoch', 0), 3), '| step =', _st.get('global_step'))
    print('best_model_checkpoint =', _st.get('best_model_checkpoint'), '| best_metric =', _st.get('best_metric'))

### Finetune (warm-start từ pretrain)
`--model_name_or_path=PRETRAIN_DIR`. Chạy in-kernel nên có progress bar.

In [ ]:
import importlib
from training import finetune
importlib.reload(finetune)
finetune.main(args_list=['configs/finetune.yaml', '--model_name_or_path', PRETRAIN_DIR])

### Upload model finetune lên HF

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_FINETUNE_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path='/content/ViSceT5/output/finetune', repo_id=HF_FINETUNE_REPO,
                  repo_type='model', ignore_patterns=['checkpoint-*', 'optimizer.pt'])
print('Uploaded finetune ->', HF_FINETUNE_REPO)